# Causal discovery
교육 분야의 중요한 과제 중 하나는 개인별 최적 학습 순서를 설계하는 것입니다.

이를 위해 실제 온라인 교육 플랫폼 Eedi의 로그를 기반으로 구축된 CausalEdu 데이터셋을 사용하여,

Discovery → Identification → Estimation → Refutation의 전 과정을 PyWhy 스택으로 구현합니다.

In [1]:
%pip -q install lingam

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings, numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 18
np.random.seed(SEED)
plt.style.use("default")

from dowhy import CausalModel
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
import lingam

## Data Setup
CausalEdu는 Eedi 플랫폼에서 수집된 관찰 데이터 및 일부 학습 개념 쌍(construct pairs)에 대해 A/B 테스트 결과와 전문가 그래프가 포함된 데이터셋입니다.

- **checkins_lessons_checkouts_training.csv**: 관찰 데이터
  - `ConstructId`는 학습 개념(예: ‘분수 덧셈’)을 나타냅니다.
  - 한 세션은 Check-in → Lesson → Check-out 순서로 진행됩니다.

- **checkin_to_checkout.csv**: A/B 테스트 결과
    - LessonConstructId → QuestionConstructId 쌍에 대해 학습 전·후 정답 패턴(`n00, n01, n10, n11`)을 제공합니다.
    - 두 가지 가설로부터 **p010_m**, **k010_m** 통계를 제공합니다.

        - 가설 1(학습은 해롭지 않다): $p010\_m = \frac{n_{01}}{n_{00}+n_{01}}$
        - 가설 2(우연 정답 보정): $k010\_m = \frac{n_{01}}{n_{00}+n_{01}+n_{10}}$

- **construct_prerequisites_test.csv**: 전문가 지식  
   - 전문가가 정의한 개념 간 선행관계(prerequisite structure)를 제공합니다.

- **construct_experiments_ates_test.csv**: A/B 테스트 결과
  - A/B 테스트을 통해 측정된 CATE로 특정 레슨 개념(`TreatmentLessonConstructId`)이 다른 개념(`QuestionConstructId`)의 미치는 실제 개입 효과를 제공합니다.

In [3]:
DATA_DIR = Path("../data/causal_edu")

training = pd.read_csv(DATA_DIR / "checkins_lessons_checkouts_training.csv")
c2c = pd.read_csv(DATA_DIR / "checkin_to_checkout.csv")
expert = pd.read_csv(DATA_DIR / "construct_prerequisites_test.csv")
ab_ates = pd.read_csv(DATA_DIR / "construct_experiments_ates_test.csv")
q_input = pd.read_csv(DATA_DIR / "construct_experiments_input_test.csv")

print("training:", training.shape)
display(training.head(3))
print("c2c:", c2c.shape)
display(c2c.head(3))
print("expert:", expert.shape)
display(expert.head(3))
print("ab_ates:", ab_ates.shape)
display(ab_ates.head(3))
print("q_input:", q_input.shape)
display(q_input.head(3))


training: (641490, 12)


,QuizSessionId,AnswerId,UserId,QuizId,QuestionId,IsCorrect,AnswerValue,CorrectAnswer,QuestionSequence,ConstructId,Type,Timestamp
0,0,0.0,0,242762,130326,1.0,4.0,4.0,1,9,Checkin,2022-02-01 01:53:31.170
1,1,1.0,1,242762,130326,1.0,4.0,4.0,1,9,Checkin,2022-02-01 02:07:31.393
2,1,2.0,1,242762,130327,0.0,1.0,4.0,2,10,Checkin,2022-02-01 02:08:27.947


c2c: (183, 9)


,LessonConstructId,QuestionConstructId,n00,n01,n10,n11,Count,p010_m,k010_m
0,70,1270,8,38,3,30,79,0.826087,0.775510
1,70,1271,56,5,6,11,78,0.081967,0.074627
2,70,1272,13,10,18,36,77,0.434783,0.243902


expert: (3277, 3)


,ConstructId,SubjectId,PrerequisiteConstructIds
0,854,33.0,{76}
1,855,33.0,{76}
2,856,33.0,"{483, 76}"


ab_ates: (88, 8)


,TreatmentLessonConstructId,QuestionConstructId,Year,ControlLessonConstructIds,ControlUsersCount,TreatmentUsersCount,ate_p_1__,ate_k_1__
0,206,211,7,{3119},73,94,0.033656,-0.019091
1,206,212,7,{3119},77,101,-0.022222,-0.036004
2,206,216,7,{3119},75,94,-0.109501,-0.118014


q_input: (45, 4)


,TreatmentLessonConstructId,QuestionConstructId,Year,ControlLessonConstructIds
0,206,211,7,{3119}
1,206,212,7,{3119}
2,206,216,7,{3119}


### 데이터 변환 (pivot)

세션 로그를 (UserId, QuizSessionId) × ConstructId 형태의 행렬로 변환해  
Discovery 구조학습 알고리즘(PC/GES/NOTEARS)이 사용할 수 있는 입력 형태로 만듭니다.

변환 과정은 다음과 같습니다:
1. `training`에서 Checkout/CheckoutRetry만 사용
2. 동일한 (User, Session, Construct) 내 마지막 시도만 선택
3. 피벗하여 (UserId, QuizSessionId)를 row index, ConstructId를 column으로 구성
4. 결측은 0으로 채우고, 시도 여부는 별도 mask로 관리

In [4]:
df = training.sort_values(["UserId", "QuizSessionId", "Timestamp"])
checkout = df[df["Type"].isin(["Checkout", "CheckoutRetry"])]
print("checkout:", checkout.shape)
display(checkout.head())

checkout: (78373, 12)


,QuizSessionId,AnswerId,UserId,QuizId,QuestionId,IsCorrect,AnswerValue,CorrectAnswer,QuestionSequence,ConstructId,Type,Timestamp
24,4,23.0,1,226964,129234,0.0,1.0,4.0,3,2480,Checkout,2022-02-01 02:23:32.213
25,4,24.0,1,226964,129234,1.0,4.0,4.0,3,2480,CheckoutRetry,2022-02-01 02:24:01.030
4231,527,3900.0,1,228600,130029,0.0,2.0,1.0,1,48,Checkout,2022-02-02 04:40:27.407
4232,527,3901.0,1,228600,130029,1.0,1.0,1.0,1,48,CheckoutRetry,2022-02-02 04:40:48.850
32,5,30.0,2,202086,104952,1.0,2.0,2.0,1,1271,Checkout,2022-02-01 04:18:31.917


In [5]:
last_checkout = (
    checkout.sort_values("Timestamp")
            .groupby(["UserId", "QuizSessionId", "ConstructId"], as_index=False)
            .tail(1)
)
print("last_checkout:", last_checkout.shape)
display(last_checkout.head())

last_checkout: (47677, 12)


,QuizSessionId,AnswerId,UserId,QuizId,QuestionId,IsCorrect,AnswerValue,CorrectAnswer,QuestionSequence,ConstructId,Type,Timestamp
25,4,24.0,1,226964,129234,1.0,4.0,4.0,3,2480,CheckoutRetry,2022-02-01 02:24:01.030
32,5,30.0,2,202086,104952,1.0,2.0,2.0,1,1271,Checkout,2022-02-01 04:18:31.917
46,6,42.0,3,202371,120911,1.0,1.0,1.0,1,1815,Checkout,2022-02-01 04:21:57.063
37,5,34.0,2,202086,104953,1.0,2.0,2.0,2,3133,CheckoutRetry,2022-02-01 04:23:00.313
57,7,52.0,4,202465,76153,0.0,3.0,1.0,3,2257,CheckoutRetry,2022-02-01 04:31:45.893


In [6]:
wide_z = last_checkout.pivot_table(
    index=["UserId", "QuizSessionId"],
    columns="ConstructId",
    values="IsCorrect",
    aggfunc="max"
).fillna(0).astype(float)

print("wide_z shape:", wide_z.shape)
display(wide_z.head(3))

wide_z shape: (35517, 1063)


ConstructId           4     9     10    20    22    26    27    28    29    \
UserId QuizSessionId                                                         
1      4               0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
       527             0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
2      5               0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   

ConstructId           30    ...  3442  3443  3450  3451  3487  3489  3510  \
UserId QuizSessionId        ...                                             
1      4               0.0  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
       527             0.0  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
2      5               0.0  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   

ConstructId           3511  3513  3515  
UserId QuizSessionId                    
1      4               0.0   0.0   0.0  
       527             0.0   0.0   0.0  
2      5               0.0   0.0   0.0  

[3 rows x 1063 columns]

In [7]:
# attempt mask 생성: '시도 여부'만 1, 나머지 0
attempt_flag = last_checkout.assign(_attempt=1).pivot_table(
    index=["UserId","QuizSessionId"],
    columns="ConstructId",
    values="_attempt",
    aggfunc="max"
).fillna(0).astype(int)

# 각 컬럼 시도율(=시도한 세션 비율) 상위 5개 확인
print("attempt_flag shape:", attempt_flag.shape)
attempt_rate = attempt_flag.mean(axis=0).rename("attempt_rate").sort_values(ascending=False)
display(attempt_rate.head().to_frame().T)


attempt_flag shape: (35517, 1063)


ConstructId,331,2908,3316,342,3343
attempt_rate,0.010361,0.00977,0.009545,0.009348,0.008306


In [8]:
# 모든 세션에서 단 한 번도 등장하지 않은 개념(Construct) 제거
#     → 학생들이 전체 기간 동안 시도조차 하지 않은 개념
allzero_cols = [c for c in wide_z.columns if (wide_z[c] == 0).all()]
print("all-zero columns:", len(allzero_cols))

wide_z_nz   = wide_z.drop(columns=allzero_cols)   if allzero_cols else wide_z
attempt_nz  = attempt_flag.drop(columns=allzero_cols) if allzero_cols else attempt_flag


all-zero columns: 7


In [9]:
# 시도율 적은 construct 제거
thr = 0.003  # 0.3%
keep_cols = [c for c, r in attempt_rate.items() if r >= thr]

wide_z_keep  = wide_z_nz[keep_cols]
attempt_keep = attempt_nz[keep_cols]

print("kept constructs:", len(keep_cols))
print("wide_z_keep:", wide_z_keep.shape)

kept constructs: 93
wide_z_keep: (35517, 93)


## Causal Discovery

`wide_z_keep`을 입력으로 인과 그래프를 학습하고,  
A/B 테스트와 전문가 그래프를 이용해 Discovery 성능을 평가합니다.  

핵심은 **학습에 사용하는 BK**와 **평가에 사용하는 Ground Truth**를 분리하는 것입니다.

### 학습 데이터 (with Background Knowledge)

- `wide_z_keep`  
- **Temporal BK**: 시간·퀴즈 순서를 거스르는 edge 금지
- c2c_bk_edges`
    - C2C에서 강한 신호(p010_m, k010_m ≥ threshold)를 가진 edge  
    - require edge로 사용해 방향성 고정
- `expert_bk_edges`
    - Expert edge 중 학습용(약 80%)

> C2C는 A/B 실험 기반이므로 Expert와 충돌 시 **C2C 우선**.

### 평가 데이터 (Ground Truth)
- `c2c_eval_edges`
    - 학습에는 사용하지 않음  
    - Discovery 평가(F1 / Precision / Recall / SHD)에만 사용
- `expert_holdout_edges` (~20%)
    - 학습에는 절대 포함하지 않음  
    - Discovery 평가용 GT로만 사용

In [10]:
# Discovery에서 실제 학습에 사용하는 노드 = wide_z_keep 컬럼
training_nodes = set(pd.to_numeric(pd.Index(wide_z_keep.columns), errors="coerce").dropna().astype(int))

print("Training nodes (wide_z_keep):", len(training_nodes))


Training nodes (wide_z_keep): 93


In [17]:
# C2C에 등장하는 노드 (Lesson + Question)
nodes_c2c = (
    set(pd.to_numeric(c2c["LessonConstructId"], errors="coerce").dropna().astype(int))
    | set(pd.to_numeric(c2c["QuestionConstructId"], errors="coerce").dropna().astype(int))
)

# Training에 포함되는 C2C 노드 / 포함되지 않는 C2C 노드
c2c_in_train  = nodes_c2c & training_nodes
c2c_out_train = nodes_c2c - training_nodes

print("C2C nodes (all):", len(nodes_c2c))
print("  ↳ in training(wide_z):", len(c2c_in_train))
print("  ↳ out of training:", len(c2c_out_train))


C2C nodes (all): 119
  ↳ in training(wide_z): 20
  ↳ out of training: 99


In [16]:
# Expert에 등장하는 노드 (ConstructId + Prerequisite)
expert_target = set(pd.to_numeric(expert["ConstructId"], errors="coerce").dropna().astype(int))

_prereq_raw   = expert["PrerequisiteConstructIds"].fillna("").astype(str).str.replace(",", "|")
_prereq_split = _prereq_raw.str.split("|")
_prereq_flat  = [t.strip() for lst in _prereq_split for t in lst if t.strip()]

expert_prereq = set(pd.to_numeric(pd.Series(_prereq_flat), errors="coerce").dropna().astype(int))
nodes_expert  = expert_target | expert_prereq

# Training에 포함되는 Expert 노드 / 포함되지 않는 Expert 노드
expert_in_train  = nodes_expert & training_nodes
expert_out_train = nodes_expert - training_nodes

print("Expert nodes (all):", len(nodes_expert))
print("  ↳ in training(wide_z):", len(expert_in_train))
print("  ↳ out of training:", len(expert_out_train))


Expert nodes (all): 3279
  ↳ in training(wide_z): 93
  ↳ out of training: 3186
